# Grid Search de Arboles Azarosos (clase 04)

Basado en `z420_ArbolesAzarosos.ipynb`. Todo lo agregado/modificado lleva el comentario `# +++`.

**Consigna (Denicolay):** Grid Search de Arboles Azarosos con **32 arbolitos constantes** y **cp = -1.0**, optimizando `feature_fraction`, `maxdepth`, `minsplit`, `minbucket`.

**Cambios respecto de z420:**
1. La ganancia del grid se mide **localmente** con particion estratificada 70/30 de 202107 (misma mecanica que `z291`: +975000 / -25000, normalizada por el 30%). No se puede submitear a Kaggle cientos de combinaciones.
2. **Paralelizacion con `parallel::mclapply` (fork)**: cada job = una combinacion de hiperparametros (32 arboles secuenciales adentro). En la e2-highmem-8 corren 8 jobs a la vez; el dataset se comparte por copy-on-write, no se duplica en memoria.
3. `set.seed` global del original NO sirve en paralelo (el orden de ejecucion cambia el stream) → cada job recibe su **propia semilla** derivada de la primigenia. Reproducible sin importar el scheduling.
4. **Checkpoints gratis**: dentro de cada job se anota la ganancia en 1, 2, 4, 8, 16 y 32 arboles (el costo es un `sum()` sobre un vector; el fit ya esta hecho). Eso llena las primeras columnas de la planilla `C4-ArbolesAzarosos` para cualquier combinacion.
5. **Resume**: se graba a disco al final de cada tanda; si la VM se corta, al relanzar saltea lo ya corrido.
6. Curva final 1→512 arboles con la mejor combinacion, fiteando los **512 arboles en paralelo** (son independientes: se guarda el vector de probs de cada uno y despues se acumulan en orden).


## 0. Librerias y threads

Maquina objetivo: GCP **e2-highmem-8** (8 vCPU / 64 GB). rpart es single-thread: el paralelismo lo ponemos nosotros a nivel de jobs.

In [1]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")  # +++ mclapply / detectCores (fork, solo Linux -- perfecto para la VM)
require("primes")    # +++ para derivar semillas por job, estilo z291

setDTthreads(percent = 100)  # +++ data.table usa todos los cores en el proceso padre (fread, joins)
options(scipen = 999)        # +++ ganancias sin notacion cientifica

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart

Loading required package: parallel

Loading required package: primes



## 1. PARAM: lo fijo y el grid

In [2]:
PARAM <- list()
PARAM$semilla_primigenia <- 271211  # +++ reemplazar por SU semilla

# +++ cuantas semillas Montecarlo por combinacion (1 alcanza para la consigna;
# +++ subir a 3-5 para desempatar los mejores si sobra tiempo de maquina)
PARAM$qsemillas <- 1L

PARAM$training_pct <- 70L  # +++ split local 70/30 estratificado, igual que z291

# +++ consigna: constantes
PARAM$num_trees <- 32L     # +++ 32 arbolitos en forma constante
PARAM$cp <- -1.0           # +++ cp fijo en -1.0 (NO va en el grid)

# +++ checkpoints intermedios dentro del grid: ganancia a 1,2,4,8,16,32 arboles
# +++ (sale gratis: el fit ya esta hecho, solo es un sum() por checkpoint)
PARAM$grabar <- c(1L, 2L, 4L, 8L, 16L, 32L)

# +++ EL GRID pedido por la consigna
# +++ granularidad: valores en escala ~log; con ensemble conviene explorar
# +++ ff bajos (decorrelacionan mas) y arboles MAS profundos que un rpart solo
PARAM$grid <- list(
  peso_baja2       = c(1.0),                     # +++ la planilla tiene la columna; agregar valores aca si se quiere gridear
  feature_fraction = c(0.10, 0.25, 0.50, 0.75),
  maxdepth         = c(6L, 8L, 10L, 12L, 14L),
  minsplit         = c(400L, 200L, 100L, 50L, 20L),
  minbucket        = c(5L, 10L, 20L, 50L, 100L)
)

PARAM$mc_cores <- detectCores()  # +++ 8 en la e2-highmem-8

## 2. Carpeta de trabajo y dataset

Deteccion de entorno: Colab (`/content`), VM GCP (`~/buckets/b1`) o repo local.

In [3]:
# +++ deteccion de entorno (Colab / VM GCP / local) en vez del setwd hardcodeado de z420
candidatos_exp <- c("/content/buckets/b1/exp", path.expand("~/buckets/b1/exp"), file.path(getwd(), "exp"))
base_exp <- candidatos_exp[dir.exists(candidatos_exp)][1]
if (is.na(base_exp)) { base_exp <- candidatos_exp[3]; dir.create(base_exp, recursive = TRUE, showWarnings = FALSE) }

experimento <- "GS4210"  # +++ Grid Search de arboles azarosos
dir.create(file.path(base_exp, experimento), showWarnings = FALSE)
setwd(file.path(base_exp, experimento))
getwd()

[1] "/home/ds/buckets/b1/exp/GS4210"

In [4]:
# +++ busco el dataset en las ubicaciones tipicas
candidatos_ds <- c(
  "/content/datasets/dataset_pequeno.csv",
  path.expand("~/datasets/dataset_pequeno.csv"),
  path.expand("~/buckets/b1/datasets/dataset_pequeno.csv")
)
archivo_dataset <- candidatos_ds[file.exists(candidatos_ds)][1]
stopifnot(!is.na(archivo_dataset))  # +++ si falla: bajar dataset_pequeno.csv a ~/datasets/

# lectura del dataset
dataset <- fread(archivo_dataset)

# trabajo solo con los datos de 202107, ultimo mes con clase_ternaria completa
# +++ (a diferencia de z420 NO uso 202109: para el grid necesito la clase verdadera
# +++  y medir ganancia local; Kaggle queda para la curva final)
dataset <- dataset[foto_mes == 202107]

invisible(gc(full = TRUE, verbose = FALSE))
nrow(dataset)
dataset[, .N, clase_ternaria]

[1] 164479

clase_ternaria,N
<chr>,<int>
CONTINUA,162077
BAJA+2,1304
BAJA+1,1098


## 3. Particion estratificada (original de z291) y precomputos compartidos

In [5]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}

In [6]:
# genero numeros primos  (igual que z291)
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia)
PARAM$semillas <- sample(primos, PARAM$qsemillas)

# +++ precomputo UNA sola vez por semilla: split, pesos y vector de ganancia del test.
# +++ z420/z291 re-particionaban adentro de cada corrida; aca lo hacemos en el padre
# +++ y los hijos del fork lo LEEN compartido (copy-on-write => 0 copias extra en RAM)
splits <- list()
for (i in seq_along(PARAM$semillas)) {
  particionar(dataset,
    division = c(PARAM$training_pct, 100L - PARAM$training_pct),
    agrupa = "clase_ternaria",
    seed = PARAM$semillas[i]
  )
  splits[[i]] <- list(
    semilla = PARAM$semillas[i],
    dtrain = dataset[fold == 1][, fold := NULL],
    dtest  = dataset[fold == 2][, fold := NULL]
  )
  # +++ vector de ganancia del test precalculado: la ganancia de un corte es
  # +++ sum(vgan[prob > umbral]) -- un solo ifelse por semilla en vez de uno por job
  splits[[i]]$vgan <- splits[[i]]$dtest[, ifelse(clase_ternaria == "BAJA+2", 975000, -25000)]
  # +++ clase del train para armar pesos si peso_baja2 != 1
  splits[[i]]$clase_train <- splits[[i]]$dtrain$clase_ternaria
}
dataset[, fold := NULL]

# Establezco cuales son los campos que puedo usar para la prediccion
campos_buenos <- copy(setdiff(colnames(dataset), c("clase_ternaria")))
length(campos_buenos)

[1] 154

## 4. El worker: un ensemble de 32 arbolitos azarosos

Es el loop central de `z420` convertido en funcion pura, con ganancia local en cada checkpoint.
El truco del umbral es el mismo del original: cortar la **suma** de probs en `k/40` equivale a cortar el **promedio** de los `k` arboles en `1/40` (por eso el promedio "ya sale solo", como discutian en el grupo).

In [7]:
# +++ corre UN job del grid: una combinacion de hiperparametros, 32 arboles
EnsembleAzaroso <- function(job) {
  setDTthreads(1)  # +++ dentro del hijo: 1 thread, ya hay 8 procesos (evita oversubscription)

  sp <- splits[[job$isem]]
  set.seed(job$semilla_arboles)  # +++ semilla PROPIA del job (no el set.seed global de z420):
                                 # +++ reproducible aunque mclapply ejecute en cualquier orden

  # +++ cp fijo por consigna; el resto viene del grid
  control <- rpart.control(
    cp = PARAM$cp,
    maxdepth = job$maxdepth,
    minsplit = job$minsplit,
    minbucket = job$minbucket,
    xval = 0
  )

  # +++ pesos solo si hace falta (peso_baja2 == 1 => NULL, rpart mas rapido)
  pesos <- NULL
  if (job$peso_baja2 != 1.0) {
    pesos <- ifelse(sp$clase_train == "BAJA+2", job$peso_baja2, 1.0)
  }

  qty_campos_a_utilizar <- as.integer(length(campos_buenos) * job$feature_fraction)

  prob_acumulada <- numeric(nrow(sp$dtest))  # +++ acumulador de prob BAJA+2 en el TEST local
  ganancias <- setNames(numeric(length(PARAM$grabar)), paste0("gan_", PARAM$grabar))
  t0 <- Sys.time()

  for (arbolito in seq_len(PARAM$num_trees)) {
    # elijo los campos al azar   (identico a z420)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    campos_random <- paste(campos_random, collapse = " + ")
    # +++ as.formula + environment(): rpart evalua weights en el environment de la
    # +++ formula (NSE); sin esto adentro de una funcion falla con 'pesos not found'
    # +++ (mismo truco que usa z291 en ArbolEstimarGanancia)
    formulita <- as.formula(paste0("clase_ternaria ~ ", campos_random))
    environment(formulita) <- environment()

    if (is.null(pesos)) {
      # +++ sin weights= : rpart no intenta evaluar nada y es mas rapido
      modelo <- rpart(formulita, data = sp$dtrain, xval = 0, control = control)
    } else {
      modelo <- rpart(formulita, data = sp$dtrain, xval = 0, control = control, weights = pesos)
    }

    # +++ me quedo SOLO con la columna BAJA+2 (z420 arrastraba la matriz de 3 clases)
    prob_acumulada <- prob_acumulada + predict(modelo, sp$dtest, type = "prob")[, "BAJA+2"]

    if (arbolito %in% PARAM$grabar) {
      # +++ ganancia local en el checkpoint:
      # +++   prob_acumulada > k/40   <=>   promedio de los k arboles > 1/40
      # +++ normalizo como z291: escalo el 30% de test a dataset completo
      gan <- sum(sp$vgan[prob_acumulada > arbolito / 40])
      ganancias[paste0("gan_", arbolito)] <-
        gan / ((100 - PARAM$training_pct) / 100)
    }
  }

  # +++ devuelvo una fila lista para la tabla de resultados (con cp para la planilla)
  data.table(
    semilla = sp$semilla,
    peso_baja2 = job$peso_baja2,
    feature_fraction = job$feature_fraction,
    cp = PARAM$cp,
    minsplit = job$minsplit,
    minbucket = job$minbucket,
    maxdepth = job$maxdepth,
    t(ganancias),
    tiempo_seg = round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1)
  )
}

## 5. La tabla de jobs: grid completo, podado

In [8]:
# +++ producto cartesiano del grid con CJ() de data.table
tb_jobs <- do.call(CJ, PARAM$grid)
cat("combinaciones brutas:", nrow(tb_jobs), "\n")

# +++ PODA (leccion de la TareaHogar02): un split genera DOS hojas;
# +++ si 2*minbucket > minsplit la combinacion es ilegal/redundante -> no se corre
tb_jobs <- tb_jobs[2 * minbucket <= minsplit]
cat("combinaciones tras la poda:", nrow(tb_jobs), "\n")

# +++ cruzo con las semillas Montecarlo (qsemillas=1 => queda igual)
tb_jobs <- tb_jobs[rep(seq_len(.N), each = PARAM$qsemillas)]
tb_jobs[, isem := rep(seq_len(PARAM$qsemillas), times = .N / PARAM$qsemillas)]
tb_jobs[, semilla := PARAM$semillas[isem]]

# +++ semilla propia de cada job para el sample() de columnas, derivada de la primigenia:
# +++ deterministico, y no depende del orden en que el scheduler ejecute los jobs
set.seed(PARAM$semilla_primigenia)
tb_jobs[, semilla_arboles := sample(primos, .N)]

cat("jobs totales:", nrow(tb_jobs), "  (arboles a fitear:", nrow(tb_jobs) * PARAM$num_trees, ")\n")

combinaciones brutas: 500 
combinaciones tras la poda: 380 
jobs totales: 380   (arboles a fitear: 12160 )


## 6. Resume: saltear lo ya corrido

In [9]:
# +++ checkpoint/resume: si el archivo existe (corrida anterior cortada),
# +++ saco de tb_jobs las combinaciones que ya tienen resultado
archivo_detalle <- "gridsearch_azaroso_detalle.txt"

clave <- c("peso_baja2", "feature_fraction", "minsplit", "minbucket", "maxdepth", "semilla")

if (file.exists(archivo_detalle)) {
  tb_grid_detalle <- fread(archivo_detalle)
  tb_jobs <- tb_jobs[!tb_grid_detalle, on = clave]
  cat("resume: ya habia", nrow(tb_grid_detalle), "resultados, quedan", nrow(tb_jobs), "jobs\n")
} else {
  tb_grid_detalle <- data.table()
}

## 7. Benchmark de UN job antes de largar todo

Mide una combinacion mediana y extrapola. Con 8 vCPU el tiempo total ~ `tiempo_job * njobs / 8`.

In [10]:
# +++ pruebo un job del medio de la tabla y estimo la corrida completa
if (nrow(tb_jobs) > 0) {
  t0 <- Sys.time()
  res_bench <- EnsembleAzaroso(tb_jobs[.N %/% 2 + 1])
  seg_job <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
  cat(sprintf(
    "1 job = %.0f seg  =>  %d jobs / %d cores  ~  %.1f horas\n",
    seg_job, nrow(tb_jobs), PARAM$mc_cores,
    seg_job * nrow(tb_jobs) / PARAM$mc_cores / 3600
  ))
  print(res_bench)
}

1 job = 69 seg  =>  380 jobs / 8 cores  ~  0.9 horas
   semilla peso_baja2 feature_fraction    cp minsplit minbucket maxdepth
     <int>      <num>            <num> <num>    <int>     <int>    <int>
1:  717427          1              0.5    -1       20         5        6
       gan_1     gan_2     gan_4     gan_8    gan_16    gan_32 tiempo_seg
       <num>     <num>     <num>     <num>     <num>     <num>      <num>
1: 439500000 435750000 535500000 547166667 521666667 536083333       68.6


## 8. LA CORRIDA: mclapply en tandas con grabado a disco

- `mc.cores = 8`: un proceso hijo por vCPU (fork: el dataset y los splits se comparten, no se copian).
- `mc.preschedule = FALSE`: balanceo dinamico — los jobs con arboles profundos y `minsplit` chico tardan mucho mas que los demas; asi ningun core queda ocioso esperando.
- Tandas de `4 * cores` jobs y `fwrite` al final de cada una: si la VM (preemptible o no) se corta, se pierde a lo sumo una tanda.

In [11]:
# +++ el loop principal paralelo (reemplaza al for secuencial de z420)
batch_size <- 4L * PARAM$mc_cores
n_batches <- ceiling(nrow(tb_jobs) / batch_size)
t_inicio <- Sys.time()

for (b in seq_len(n_batches)) {
  idx <- ((b - 1L) * batch_size + 1L):min(b * batch_size, nrow(tb_jobs))

  resultados <- mclapply(
    idx,
    function(i) tryCatch(EnsembleAzaroso(tb_jobs[i]), error = function(e) NULL),  # +++ un job roto no tira la tanda
    mc.cores = PARAM$mc_cores,
    mc.preschedule = FALSE   # +++ balanceo dinamico de carga
  )

  tb_grid_detalle <- rbindlist(c(list(tb_grid_detalle), resultados), use.names = TRUE, fill = TRUE)

  # +++ grabo TODO al final de cada tanda => resume barato
  fwrite(tb_grid_detalle, file = archivo_detalle, sep = "\t")

  transcurrido <- as.numeric(difftime(Sys.time(), t_inicio, units = "mins"))
  cat(sprintf(
    "tanda %d/%d | %d resultados | %.1f min | ETA restante ~ %.1f min\n",
    b, n_batches, nrow(tb_grid_detalle), transcurrido,
    transcurrido / b * (n_batches - b)
  ))
  flush.console()
}

cat("listo:", nrow(tb_grid_detalle), "resultados\n")

tanda 1/12 | 32 resultados | 2.4 min | ETA restante ~ 26.8 min
tanda 2/12 | 64 resultados | 5.7 min | ETA restante ~ 28.3 min
tanda 3/12 | 96 resultados | 9.7 min | ETA restante ~ 29.2 min
tanda 4/12 | 128 resultados | 14.4 min | ETA restante ~ 28.9 min
tanda 5/12 | 160 resultados | 21.0 min | ETA restante ~ 29.5 min
tanda 6/12 | 192 resultados | 29.0 min | ETA restante ~ 29.0 min
tanda 7/12 | 224 resultados | 37.6 min | ETA restante ~ 26.8 min
tanda 8/12 | 256 resultados | 50.0 min | ETA restante ~ 25.0 min
tanda 9/12 | 288 resultados | 64.4 min | ETA restante ~ 21.5 min
tanda 10/12 | 320 resultados | 76.9 min | ETA restante ~ 15.4 min
tanda 11/12 | 352 resultados | 94.7 min | ETA restante ~ 8.6 min
tanda 12/12 | 380 resultados | 115.0 min | ETA restante ~ 0.0 min
listo: 380 resultados


## 9. Resultados: ranking y fila para la planilla

In [12]:
# +++ resumen por combinacion: promedio sobre semillas (con qsemillas=1 es identidad)
tb_grid <- tb_grid_detalle[,
  c(lapply(.SD, mean), list(qty = .N, tiempo_seg = sum(tiempo_seg))),
  by = .(peso_baja2, feature_fraction, cp, minsplit, minbucket, maxdepth),
  .SDcols = paste0("gan_", PARAM$grabar)
]

setorder(tb_grid, -gan_32)  # +++ ordeno por la ganancia del ensemble de 32 (la de la consigna)
tb_grid[, id := .I]
fwrite(tb_grid, file = "gridsearch_azaroso.txt", sep = "\t")

# los 10 mejores
tb_grid[1:10]

peso_baja2,feature_fraction,cp,minsplit,minbucket,maxdepth,gan_1,gan_2,gan_4,gan_8,gan_16,gan_32,qty,tiempo_seg,id
<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
1,0.50,-1,20,5,8,478416667,481666667,516583333,525000000,528500000,564250000,1,141.8,1
1,0.25,-1,200,10,8,414250000,465666667,495666667,518916667,540666667,561500000,1,75.6,2
1,0.25,-1,50,20,10,410500000,439666667,459250000,537250000,525083333,560916667,1,98.4,3
1,0.50,-1,400,50,8,454333333,470250000,518166667,538250000,549166667,560666667,1,146.7,4
1,0.25,-1,200,5,6,479833333,505750000,513916667,536250000,536000000,557000000,1,62.5,5
1,0.50,-1,200,100,12,417083333,458500000,504916667,550250000,560333333,556083333,1,193.7,6
1,0.25,-1,50,20,6,470500000,500916667,511416667,512416667,548916667,554083333,1,60.5,7
1,0.25,-1,100,10,8,356583333,395666667,454083333,495916667,533666667,553666667,1,81.0,8
1,0.25,-1,20,10,8,479833333,507500000,507833333,509166667,551166667,553416667,1,81.0,9


In [13]:
# +++ efecto marginal de cada hiperparametro (para el analisis estilo Zulip:
# +++ que valores son siempre malos, que region conviene, etc.)
for (hp in c("feature_fraction", "maxdepth", "minsplit", "minbucket")) {
  cat("\n===", hp, "===\n")
  print(tb_grid[, .(gan_32_media = mean(gan_32), gan_32_max = max(gan_32)), keyby = hp])
}


=== feature_fraction ===
Key: <feature_fraction>
   feature_fraction gan_32_media gan_32_max
              <num>        <num>      <num>
1:             0.10    515748246  540916667
2:             0.25    535639474  561500000
3:             0.50    531688596  564250000
4:             0.75    513751754  547250000

=== maxdepth ===
Key: <maxdepth>
   maxdepth gan_32_media gan_32_max
      <int>        <num>      <num>
1:        6    521426535  557000000
2:        8    532057018  564250000
3:       10    526048246  560916667
4:       12    523513158  556083333
5:       14    517990132  547250000

=== minsplit ===
Key: <minsplit>
   minsplit gan_32_media gan_32_max
      <int>        <num>      <num>
1:       20    518589583  564250000
2:       50    524072222  560916667
3:      100    525032292  553666667
4:      200    524666667  561500000
5:      400    525415000  560666667

=== minbucket ===
Key: <minbucket>
   minbucket gan_32_media gan_32_max
       <int>        <num>      <num>
1:  

In [14]:
# +++ FILA para la planilla 'C4 - Grid Search ArbolesAzaroso':
# +++ estudiante | semilla primigenia | tiempo de corrida | ff | cp | maxdepth | minsplit | minbucket | ganancia_mean 32 arboles
mejor <- tb_grid[1]
cat("semilla primigenia :", PARAM$semilla_primigenia, "\n")
cat("tiempo de corrida  :", round(sum(tb_grid$tiempo_seg) / 60), "min (suma de jobs; wall-clock ~ /8)\n")
cat("feature_fraction   :", mejor$feature_fraction, "\n")
cat("cp                 :", mejor$cp, "\n")
cat("maxdepth           :", mejor$maxdepth, "\n")
cat("minsplit           :", mejor$minsplit, "\n")
cat("minbucket          :", mejor$minbucket, "\n")
cat("ganancia_mean 32   :", mejor$gan_32, "\n")

semilla primigenia : 271211 
tiempo de corrida  : 886 min (suma de jobs; wall-clock ~ /8)
feature_fraction   : 0.5 
cp                 : -1 
maxdepth           : 8 
minsplit           : 20 
minbucket          : 5 
ganancia_mean 32   : 564250000 


## 10. Curva 1 → 512 arboles con la mejor combinacion

Para la fila de la planilla `C4-ArbolesAzarosos` (columnas 1, 2, 4, ..., 512).

Truco de paralelizacion: los arboles del ensemble son **independientes** — se fitean los 512 en paralelo, cada hijo devuelve su vector de probs sobre el test, y despues se acumulan **en orden** en el padre. La matriz de probs (test ~50k filas x 512) ocupa ~200 MB: nada para 64 GB.

In [15]:
# +++ curva local con la mejor combinacion del grid
PARAM$num_trees_max <- 512L
grabar <- c(1, 2, 4, 8, 16, 32, 64, 128, 256, 384, 512)   # igual que z420

sp <- splits[[1]]
control_mejor <- rpart.control(
  cp = PARAM$cp, maxdepth = mejor$maxdepth,
  minsplit = mejor$minsplit, minbucket = mejor$minbucket, xval = 0
)
qty_campos <- as.integer(length(campos_buenos) * mejor$feature_fraction)

# +++ una semilla por arbol, derivada de la primigenia => reproducible en paralelo
set.seed(PARAM$semilla_primigenia)
semillas_arbol <- sample(primos, PARAM$num_trees_max)

# +++ fiteo los 512 arboles EN PARALELO (cada uno devuelve su vector de prob BAJA+2)
lista_probs <- mclapply(seq_len(PARAM$num_trees_max), function(i) {
  setDTthreads(1)
  set.seed(semillas_arbol[i])
  campos_random <- paste(sample(campos_buenos, qty_campos), collapse = " + ")
  modelo <- rpart(paste0("clase_ternaria ~ ", campos_random),
    data = sp$dtrain, xval = 0, control = control_mejor
  )
  p <- predict(modelo, sp$dtest, type = "prob")[, "BAJA+2"]
  if (i %% 32L == 0L) cat("  arbol", i, "/", PARAM$num_trees_max, "listo\n")  # +++ progreso en terminal
  p
}, mc.cores = PARAM$mc_cores, mc.preschedule = FALSE)

# +++ acumulo en orden y calculo la ganancia en cada checkpoint (todo vectorizado)
prob_acumulada <- numeric(nrow(sp$dtest))
tb_curva <- data.table(arbolitos = integer(), ganancia = numeric())
for (arbolito in seq_len(PARAM$num_trees_max)) {
  prob_acumulada <- prob_acumulada + lista_probs[[arbolito]]
  if (arbolito %in% grabar) {
    gan <- sum(sp$vgan[prob_acumulada > arbolito / 40]) / ((100 - PARAM$training_pct) / 100)
    tb_curva <- rbind(tb_curva, data.table(arbolitos = arbolito, ganancia = gan))
  }
}

fwrite(tb_curva, file = "curva_arbolitos.txt", sep = "\t")
tb_curva  # +++ estos valores llenan la fila de 'C4-ArbolesAzarosos' (columnas 1..512)

arbolitos,ganancia
<int>,<dbl>
1,399000000
2,462583333
4,493083333
8,504833333
16,508083333
32,529500000
64,535750000
128,542166667
256,538166667


## 11. Version Kaggle de la curva, con submits LOCALIZADOS

Se entrena con **todo 202107**, se predice **202109** y se genera un CSV por checkpoint (igual que z420, pero con los arboles fiteados en paralelo).

Para cuidar el limite diario de Kaggle, se submitean automaticamente **solo** los checkpoints elegidos en `PARAM$submit_checkpoints` (los demas CSV quedan generados en la carpeta del experimento por si despues queres submitear alguno a mano).

In [16]:
CORRER_KAGGLE <- TRUE  # +++ poner FALSE para saltear esta seccion

# +++ submits LOCALIZADOS: solo estos checkpoints se suben automaticamente a Kaggle
# +++ (elegir pocos puntos: el limite diario no perdona; el resto de los CSV queda en disco)
PARAM$submit_checkpoints <- c(32L, 128L, 512L)

if (CORRER_KAGGLE) {
  dataset_full <- fread(archivo_dataset)
  dtrain_full <- dataset_full[foto_mes == 202107]
  dfuture <- dataset_full[foto_mes == 202109]
  dfuture[, clase_ternaria := NA]   # igual que z420

  set.seed(PARAM$semilla_primigenia)
  semillas_arbol_k <- sample(primos, PARAM$num_trees_max)

  # +++ mismos hiperparametros ganadores, arboles en paralelo
  cat("fiteando", PARAM$num_trees_max, "arboles en paralelo sobre 202107 completo...\n")
  flush.console()
  lista_probs_k <- mclapply(seq_len(PARAM$num_trees_max), function(i) {
    setDTthreads(1)
    set.seed(semillas_arbol_k[i])
    campos_random <- paste(sample(campos_buenos, qty_campos), collapse = " + ")
    modelo <- rpart(paste0("clase_ternaria ~ ", campos_random),
      data = dtrain_full, xval = 0, control = control_mejor
    )
    p <- predict(modelo, dfuture, type = "prob")[, "BAJA+2"]
    if (i %% 32L == 0L) cat("  arbol", i, "/", PARAM$num_trees_max, "listo\n")  # +++ progreso en terminal
    p
  }, mc.cores = PARAM$mc_cores, mc.preschedule = FALSE)

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob_acumulada := 0]

  for (arbolito in seq_len(PARAM$num_trees_max)) {
    tb_prediccion[, prob_acumulada := prob_acumulada + lista_probs_k[[arbolito]]]
    if (arbolito %in% grabar) {
      umbral_corte <- (1 / 40) * arbolito           # igual que z420
      tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]
      archivo_kaggle <- paste0("KA420_", sprintf("%.3d", arbolito), ".csv")  # +++ prefijo KA420
      fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
        file = archivo_kaggle, sep = ","
      )
      cat("checkpoint", arbolito, "->", archivo_kaggle,
          "( estimulados =", tb_prediccion[, sum(Predicted)], ")\n")  # +++ progreso en terminal

      if (arbolito %in% PARAM$submit_checkpoints) {
        # +++ submit automatico SOLO en los checkpoints localizados (igual que z420)
        comando <- "kaggle competitions submit"
        competencia <- "-c labo-1-ba-inicial"
        arch <- paste("-f", archivo_kaggle)
        mensaje <- paste0("-m 'azarosos ", arbolito, " arboles ff=", mejor$feature_fraction,
          " cp=", PARAM$cp, " md=", mejor$maxdepth,
          " ms=", mejor$minsplit, " mb=", mejor$minbucket, "'")
        linea <- paste(comando, competencia, arch, mensaje)
        salida <- system(linea, intern = TRUE)
        cat("  SUBMIT:", salida, "\n")  # +++ resultado del submit en terminal
      }
      flush.console()
    }
  }
}

fiteando 512 arboles en paralelo sobre 202107 completo...
checkpoint 1 -> KA420_001.csv ( estimulados = 7643 )
checkpoint 2 -> KA420_002.csv ( estimulados = 7972 )
checkpoint 4 -> KA420_004.csv ( estimulados = 9845 )
checkpoint 8 -> KA420_008.csv ( estimulados = 10417 )
checkpoint 16 -> KA420_016.csv ( estimulados = 10532 )
checkpoint 32 -> KA420_032.csv ( estimulados = 10402 )
  SUBMIT: 99 submissions remaining today. Successfully submitted to Labo 1 BA Inicial 
checkpoint 64 -> KA420_064.csv ( estimulados = 10289 )
checkpoint 128 -> KA420_128.csv ( estimulados = 10248 )
  SUBMIT: 98 submissions remaining today. Successfully submitted to Labo 1 BA Inicial 
checkpoint 256 -> KA420_256.csv ( estimulados = 10340 )
checkpoint 384 -> KA420_384.csv ( estimulados = 10303 )
checkpoint 512 -> KA420_512.csv ( estimulados = 10368 )
  SUBMIT: 97 submissions remaining today. Successfully submitted to Labo 1 BA Inicial 


In [17]:
# +++ ver las ganancias de los submits directo en la terminal
# +++ (Kaggle tarda unos segundos en puntuar: si sale 'pending', re-ejecutar esta celda)
if (CORRER_KAGGLE) {
  cat(system("kaggle competitions submissions -c labo-1-ba-inicial", intern = TRUE), sep = "\n")
}

     ref  fileName           date                        description                                        status                     publicScore  privateScore  
--------  -----------------  --------------------------  -------------------------------------------------  -------------------------  -----------  ------------  
56312831  KA420_512.csv      2026-09-17 20:01:35.063000  azarosos 512 arboles ff=0.5 cp=-1 md=8 ms=20 mb=5  SubmissionStatus.COMPLETE  344.081                    
56312830  KA420_128.csv      2026-09-17 20:01:32.343000  azarosos 128 arboles ff=0.5 cp=-1 md=8 ms=20 mb=5  SubmissionStatus.COMPLETE  336.581                    
56312829  KA420_032.csv      2026-09-17 20:01:29.937000  azarosos 32 arboles ff=0.5 cp=-1 md=8 ms=20 mb=5   SubmissionStatus.COMPLETE  350.748                    
56193226  KA420_512.csv      2026-09-12 21:31:49.720000  cp=-1  minsplit=50  minbucket=20 maxdepth=6        SubmissionStatus.COMPLETE  344.081                    
56192182  KA420_384.cs

## 12. Que va en cada planilla

**`C4 - Grid Search ArbolesAzaroso`** (una fila):
- semilla primigenia, tiempo de corrida (wall-clock), y de la mejor combinacion: `feature_fraction`, `cp = -1`, `maxdepth`, `minsplit`, `minbucket`, `ganancia_mean 32 arboles` → salida de la celda de la seccion 9.

**`C4-ArbolesAzarosos`** (fila con la curva):
- `peso_baja2`, `feature_fraction`, `cp`, `minsplit`, `minbucket`, `maxdepth` de la mejor combinacion + las ganancias por cantidad de arbolitos (1, 2, 4, ..., 512) → `tb_curva` de la seccion 10 (o los resultados de Kaggle de la seccion 11 si eso es lo que pide la catedra).

Referencia de sanidad: la fila ya cargada (Rodriguez) con `ff=0.25, cp=-0.5, minsplit=200, minbucket=71, maxdepth=16` da **593.666.667** en 32 arboles: la mejor combinacion del grid deberia estar en ese orden de magnitud o arriba.